# Speech Utils (su) Demo

This notebook demonstrates the key features of the `su` speech utilities package.

**Note**: If you encounter TTS errors on macOS, this is a known pyttsx3 issue. The package includes workarounds, but you may need to restart the kernel if issues persist.

In [1]:
# Import the speech utilities
import su
from functools import partial
import numpy as np

## Speak: Text-to-Speech

Let's start with simple text-to-speech functionality.

In [2]:
# Basic speech
print("Speaking: Hello, world!")
try:
    su.speak("Hello, world!")
    print("✓ Basic speech successful")
except Exception as e:
    print(f"⚠️ TTS Error: {e}")
    print("This may be a macOS pyttsx3 issue. Try restarting the kernel.")

Speaking: Hello, world!
✓ Basic speech successful


In [3]:
# Custom voice settings
print("Speaking with custom settings...")
try:
    su.speak("This is slower and quieter", rate=120, volume=0.6)
    print("✓ Custom voice settings successful")
except Exception as e:
    print(f"⚠️ TTS Error: {e}")

Speaking with custom settings...
✓ Custom voice settings successful


In [4]:
# Save speech to file (alternative approach if live speech fails)
print("Saving speech to file...")
try:
    su.speak("This will be saved to a file", save_to="demo_output.wav")
    print("✓ Speech saved to demo_output.wav")
except Exception as e:
    print(f"⚠️ Save to file error: {e}")
    print("Trying alternative approach...")
    
    # Alternative: Use the class directly
    try:
        tts = su.TextToSpeech(rate=200, volume=0.9)
        tts.speak("Alternative approach", save_to="demo_output_alt.wav")
        print("✓ Alternative save successful")
    except Exception as e2:
        print(f"⚠️ Alternative also failed: {e2}")
        print("You may need to restart the Python kernel")

Saving speech to file...
✓ Speech saved to demo_output.wav


## Transcribe: Speech-to-Text

In [14]:
# Test transcription with various audio sources
print("=== Transcription Examples ===")

# 1. Test with non-existent file (error handling)
print("\n1. Testing error handling with non-existent file:")
result = su.transcribe("nonexistent_audio.wav")
print(f"Result: {result}")

# 2. Test microphone transcription (same as recognize)
print("\n2. Microphone transcription (using dict format):")
print("This would listen to microphone for 3 seconds...")
result = su.transcribe({'type': 'microphone', 'timeout': 3})

# 3. Generate audio bytes and transcribe them back
print("\n3. Round-trip test: Text → Audio → Text")
original_text = "The quick brown fox jumps over the lazy dog"
print(f"Original text: '{original_text}'")

try:
    # Generate audio bytes from text
    audio_bytes = su.speak(original_text, egress=lambda x: x, send_to_speakers=False)
    print(f"Generated {len(audio_bytes)} bytes of audio")
    
    # Transcribe the audio bytes back to text
    from io import BytesIO
    recovered_text = su.transcribe(BytesIO(audio_bytes))
    print(f"Recovered text: '{recovered_text}'")
    
    if recovered_text:
        # Compare texts (case insensitive)
        match = original_text.lower() == recovered_text.lower()
        print(f"Perfect match: {match}")
        if not match:
            print("Note: Some variation is normal due to TTS/ASR processing")
    else:
        print("Transcription failed - this is common without proper audio setup")
        
except Exception as e:
    print(f"Round-trip test failed: {e}")
    print("This is expected if TTS has issues")

=== Transcription Examples ===

1. Testing error handling with non-existent file:
Result: None

2. Microphone transcription (using dict format):
This would listen to microphone for 3 seconds...

3. Round-trip test: Text → Audio → Text
Original text: 'The quick brown fox jumps over the lazy dog'
Generated 83264 bytes of audio
Recovered text: 'None'
Transcription failed - this is common without proper audio setup
Generated 83264 bytes of audio
Recovered text: 'None'
Transcription failed - this is common without proper audio setup


In [ ]:
# More advanced transcription examples
print("=== Advanced Audio Source Examples ===")

# 4. Create multiple audio files with different text
test_phrases = [
    "Hello world, this is test one",
    "Python is a great programming language",
    "Speech recognition and text to speech are fascinating"
]

print("\n4. Creating and transcribing multiple audio files:")
audio_files = []

for i, phrase in enumerate(test_phrases):
    filename = f"test_audio_{i+1}.wav"
    try:
        # Create audio file
        su.speak(phrase, egress=filename, send_to_speakers=False)
        audio_files.append(filename)
        print(f"✓ Created {filename}")
    except Exception as e:
        print(f"✗ Failed to create {filename}: {e}")

# Transcribe each file
print("\nTranscribing audio files:")
for filename in audio_files:
    try:
        # Transcribe from file path
        result = su.transcribe(filename)
        print(f"File {filename}: '{result}'")
    except Exception as e:
        print(f"Failed to transcribe {filename}: {e}")

# 5. Demonstrate file-like object transcription
print("\n5. File-like object transcription:")
if audio_files:
    try:
        # Read audio file as bytes and transcribe
        with open(audio_files[0], 'rb') as f:
            audio_bytes = f.read()
        
        print(f"Read {len(audio_bytes)} bytes from {audio_files[0]}")
        
        # Transcribe from bytes
        result = su.transcribe(audio_bytes)
        print(f"Transcription from bytes: '{result}'")
        
        # Transcribe from BytesIO
        from io import BytesIO
        audio_stream = BytesIO(audio_bytes)
        result = su.transcribe(audio_stream)
        print(f"Transcription from BytesIO: '{result}'")
        
    except Exception as e:
        print(f"File-like object test failed: {e}")

## Text-to-Speech-to-Text Test

In [2]:
import su 

original_text = "The quick brown fox jumps over the lazy dog"
print(f"Original text: '{original_text}'")

try:
    # Generate audio bytes
    audio_bytes = su.speak(original_text, egress=lambda x: x, send_to_speakers=False)
    print(f"Generated {len(audio_bytes)} bytes of audio")
    
    # Transcribe back to text
    transcribed_text = su.transcribe(audio_bytes)
    print(f"Transcribed text: '{transcribed_text}'")
    
    # Check if transcription succeeded
    if transcribed_text is None:
        print("❌ Transcription failed - returned None")
        print("This is common when:")
        print("  - Speech recognition engine is not available")
        print("  - Audio quality is poor")
        print("  - Network issues (for online engines)")
        print("  - Audio format not supported")
    else:
        # Compare texts (case insensitive, stripped)
        original_clean = original_text.lower().strip()
        transcribed_clean = transcribed_text.lower().strip()
        
        if original_clean == transcribed_clean:
            print("✅ Perfect match!")
        else:
            print("⚠️ Slight mismatch (this is normal for TTS→ASR)")
            print(f"  Original:    '{original_clean}'")
            print(f"  Transcribed: '{transcribed_clean}'")
            
            # Check if they're similar enough
            similarity = len(set(original_clean.split()) & set(transcribed_clean.split()))
            total_words = len(set(original_clean.split()) | set(transcribed_clean.split()))
            similarity_ratio = similarity / total_words if total_words > 0 else 0
            
            print(f"  Word similarity: {similarity_ratio:.2%}")
            
            if similarity_ratio > 0.8:
                print("✅ Close enough - good TTS→ASR pipeline!")
            else:
                print("❌ Significant difference - may indicate issues")

except Exception as e:
    print(f"❌ Test failed with error: {e}")
    print("This could be due to:")
    print("  - TTS engine issues (pyttsx3 problems)")
    print("  - Missing dependencies")
    print("  - Audio system problems")

Original text: 'The quick brown fox jumps over the lazy dog'
Generated 57910 bytes of audio
Generated 57910 bytes of audio
Transcribed text: 'the quick brown fox jumps over the lazy'
⚠️ Slight mismatch (this is normal for TTS→ASR)
  Original:    'the quick brown fox jumps over the lazy dog'
  Transcribed: 'the quick brown fox jumps over the lazy'
  Word similarity: 87.50%
✅ Close enough - good TTS→ASR pipeline!
Transcribed text: 'the quick brown fox jumps over the lazy'
⚠️ Slight mismatch (this is normal for TTS→ASR)
  Original:    'the quick brown fox jumps over the lazy dog'
  Transcribed: 'the quick brown fox jumps over the lazy'
  Word similarity: 87.50%
✅ Close enough - good TTS→ASR pipeline!


## Partial Application for Voice Profiles

Create custom voice functions using partial application.

**Note**: If TTS is having issues, these examples will be commented out for demonstration.

In [4]:
from functools import partial

# Create voice profiles (with error handling)
def safe_speak(text, **kwargs):
    """Wrapper for speak with error handling."""
    try:
        su.speak(text, **kwargs)
        return True
    except Exception as e:
        print(f"⚠️ TTS Error: {e}")
        return False

# Create voice profiles
robot_voice = partial(safe_speak, rate=300, volume=1.0)
narrator_voice = partial(safe_speak, rate=150, volume=0.7)
whisper_voice = partial(safe_speak, rate=100, volume=0.4)

print("Testing voice profiles:")
print("Robot voice:")
robot_voice("SYSTEM INITIALIZED. READY FOR COMMANDS.")

print("Narrator voice:")
narrator_voice("Once upon a time, in a land far away...")

print("Whisper voice:")
whisper_voice("This is a secret message.")

Testing voice profiles:
Robot voice:
Narrator voice:
Narrator voice:
Whisper voice:
Whisper voice:


True

## Advanced TextToSpeech Class

For more control, use the TextToSpeech class directly.

In [6]:
try:
    # Create TTS instance
    tts = su.TextToSpeech(rate=180, volume=0.8)

    # Get available voices
    voices = tts.get_voices()
    print(f"Available voices: {len(voices)}")
    for i, voice in enumerate(voices[:3]):  # Show first 3 voices
        print(f"  {i+1}. {voice['name']} - {voice['lang']}")

    # Speak with current voice
    print("Testing default voice...")
    tts.speak("This is the default voice.")
    print("✓ Voice test successful")
    
except Exception as e:
    print(f"⚠️ TTS Class Error: {e}")
    print("Showing voice info without speaking:")
    
    # Still show voice information even if speaking fails
    try:
        tts = su.TextToSpeech()
        voices = tts.get_voices()
        print(f"Available voices: {len(voices)}")
        for i, voice in enumerate(voices[:3]):
            print(f"  {i+1}. {voice['name']} - {voice['lang']}")
    except Exception as e2:
        print(f"⚠️ Could not get voice info: {e2}")

Available voices: 177
  1. Albert - ['en_US']
  2. Alice - ['it_IT']
  3. Alva - ['sv_SE']
Testing default voice...
✓ Voice test successful


## Speech Recognition (Note: Requires microphone)

These examples show speech recognition functionality. They're commented out since they require microphone access.

In [8]:
# Speech recognition examples (uncomment to try)
# Note: These require microphone access and may prompt for permissions

# Basic recognition
print("Listening for 3 seconds...")
text = su.recognize(timeout=3)
print(f"You said: {text}")


Listening for 3 seconds...
You said: do you have access to the microphone just


In [10]:
# Custom recognition with different engine
offline_recognize = partial(su.recognize, engine='sphinx')
print("Using offline recognition...")
text = offline_recognize(timeout=3)
print(f"Offline recognition result: {text}")

Using offline recognition...
Offline recognition result: None


## Audio Processing

Demonstrate audio file processing capabilities.

In [11]:
# Audio processing examples
# Note: These would work with actual audio files

# Example of what you could do with audio files:
print("Audio processing capabilities:")
print("- Load audio files: su.AudioProcessor.load_audio('file.wav')")
print("- Convert formats: su.AudioProcessor.convert_format('input.mp3', 'output.wav')")
print("- Extract features: su.AudioProcessor.extract_features(audio, sample_rate)")
print("- Transcribe files: su.transcribe('audio.wav')")

Audio processing capabilities:
- Load audio files: su.AudioProcessor.load_audio('file.wav')
- Convert formats: su.AudioProcessor.convert_format('input.mp3', 'output.wav')
- Extract features: su.AudioProcessor.extract_features(audio, sample_rate)
- Transcribe files: su.transcribe('audio.wav')


## Error Handling and Robustness

The su package handles various edge cases gracefully.

In [ ]:
# Test error handling
print("Testing error handling...")

# Try to transcribe a non-existent file
result = su.transcribe("nonexistent_file.wav")
print(f"Transcription of non-existent file: {result}")

# Recognition with very short timeout (will likely return None)
# result = su.recognize(timeout=0.1)
# print(f"Very short timeout recognition: {result}")

print("Error handling works correctly!")

## Voice Assistant Example

Here's a simple voice assistant implementation.

In [4]:
from typing import Callable
import su

def default_response_func(text: str) -> str:
    """Default response function that echoes the input text."""
    response = f"You said: {text}"
    print(response)
    return response

def voice_assistant(
        response_func: Callable[[str], None] = default_response_func, 
        *, 
        listen_time: int = 5,
        safe_word: str = "goodbye"
    ):
    """
    A simple voice assistant that echoes what you say.
    Uncomment the code inside to try it.
    """
    print("Voice assistant example (commented out for safety)")
    print("This would:")
    print("1. Listen for your voice")
    print("2. Convert speech to text") 
    print("3. Respond by speaking the text back")
    print("4. Exit when you say 'goodbye'")
    
    assistant_voice = partial(su.speak, rate=180, volume=0.8)
    quick_listen = partial(su.recognize, timeout=listen_time, engine='google')
    
    assistant_voice("Hello! I'm your voice assistant. Say something!")
    
    while True:
        try:
            text = quick_listen()
            
            if text:
                response = response_func(text)
                assistant_voice(response)
                
                if safe_word in text.lower():
                    assistant_voice("Goodbye!")
                    break
            else:
                print("No speech detected")
        except KeyboardInterrupt:
            print("Exiting voice assistant")
            break



In [14]:
voice_assistant()

Voice assistant example (commented out for safety)
This would:
1. Listen for your voice
2. Convert speech to text
3. Respond by speaking the text back
4. Exit when you say 'goodbye'
You said: something
You said: something
You said: something again
You said: something again
You said: something again
You said: something again
You said: now I say goodbye
You said: now I say goodbye


In [5]:
import oa 
from functools import partial

ai_voice_assistant = partial(voice_assistant, response_func=oa.chat, listen_time=5, safe_word="goodbye")

In [6]:
ai_voice_assistant()

Voice assistant example (commented out for safety)
This would:
1. Listen for your voice
2. Convert speech to text
3. Respond by speaking the text back
4. Exit when you say 'goodbye'
Exiting voice assistant
Exiting voice assistant


## Summary

The `su` package provides:

1. **Simple Functions**: `su.speak()`, `su.recognize()`, `su.transcribe()`
2. **Customization**: Keyword arguments for rate, volume, engine selection
3. **Partial Application**: Create custom voice profiles and recognition functions
4. **Robust Classes**: Full-featured `SpeechRecognizer`, `TextToSpeech`, `AudioProcessor`
5. **Error Handling**: Graceful handling of missing files, hardware issues, etc.

Perfect for building voice interfaces, audio processing pipelines, and speech applications!

**Troubleshooting TTS Issues on macOS:**
- If you encounter pyttsx3 errors, try restarting the Python kernel
- Use the `save_to` parameter to save audio files instead of direct playback
- The package includes workarounds for known macOS TTS bugs
- Consider using alternative TTS libraries for production applications

# Appendix

## Troubleshooting transcribe

In [1]:
import su
from io import BytesIO

print("=== Debugging Transcription Issues ===")

original_text = "Hello world"
print(f"Original text: '{original_text}'")

# Generate audio bytes
try:
    audio_bytes = su.speak(original_text, egress=lambda x: x, send_to_speakers=False)
    print(f"Generated {len(audio_bytes)} bytes of audio")
    
    # Save to file first to test
    with open("debug_audio.wav", "wb") as f:
        f.write(audio_bytes)
    print("✓ Saved audio to debug_audio.wav")
    
    # Try transcribing from file path first
    print("\n1. Testing transcription from file path:")
    result_file = su.transcribe("debug_audio.wav")
    print(f"File transcription result: '{result_file}'")
    
    # Try transcribing from bytes
    print("\n2. Testing transcription from bytes:")
    result_bytes = su.transcribe(audio_bytes)
    print(f"Bytes transcription result: '{result_bytes}'")
    
    # Try transcribing from BytesIO
    print("\n3. Testing transcription from BytesIO:")
    audio_stream = BytesIO(audio_bytes)
    result_stream = su.transcribe(audio_stream)
    print(f"Stream transcription result: '{result_stream}'")
    
    # Test with different engines
    print("\n4. Testing with sphinx engine:")
    result_sphinx = su.transcribe(audio_bytes, engine='sphinx')
    print(f"Sphinx result: '{result_sphinx}'")
    
    # Check if the audio file is valid
    print("\n5. Audio file analysis:")
    try:
        import wave
        with wave.open("debug_audio.wav", 'rb') as wav_file:
            frames = wav_file.getnframes()
            sample_rate = wav_file.getframerate()
            duration = frames / sample_rate
            print(f"Audio info: {duration:.2f}s, {sample_rate}Hz, {frames} frames")
    except Exception as e:
        print(f"Could not analyze audio file: {e}")
    
    # Try using speech_recognition directly
    print("\n6. Direct speech_recognition test:")
    try:
        import speech_recognition as sr
        r = sr.Recognizer()
        with sr.AudioFile("debug_audio.wav") as source:
            audio = r.record(source)
            text = r.recognize_google(audio)
            print(f"Direct SR result: '{text}'")
    except Exception as e:
        print(f"Direct SR failed: {e}")
        
except Exception as e:
    print(f"Test failed: {e}")
    import traceback
    traceback.print_exc()

=== Debugging Transcription Issues ===
Original text: 'Hello world'
Generated 20104 bytes of audio
✓ Saved audio to debug_audio.wav

1. Testing transcription from file path:
File transcription result: 'hello'

2. Testing transcription from bytes:
Bytes transcription result: 'hello'

3. Testing transcription from BytesIO:
Stream transcription result: 'hello'

4. Testing with sphinx engine:
Sphinx result: 'None'

5. Audio file analysis:
Audio info: 0.63s, 16000Hz, 10013 frames

6. Direct speech_recognition test:
Direct SR result: 'hello'
